# Step 1: Overview
# -------------------------------------------
# This script processes an EPC dataset to model PV panel deployment.
#
# It creates two new columns:
# 1. MODELED_PV:
#    - If EXISTING_PV_PANELS > 0 → use EXISTING_PV_PANELS
#    - If EXISTING_PV_PANELS == 0 → use POSSIBLE_PV_PANELS
#
# 2. PV_UPGRADE:
#    - 0 if PV already exists
#    - 1 if no PV exists (i.e., upgrade opportunity)
#
# The updated dataset is then saved to a new CSV file.

In [1]:
# Step 2: Import Required Libraries
# -------------------------------------------
import pandas as pd

In [2]:
# Step 3: Load Dataset
# -------------------------------------------
file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_existing_pv.csv"

df = pd.read_csv(file_path)

# Preview to confirm load
df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,INSPECTION_DATE,TYPE_OF_ASSESSMENT,...,USABLE_ROOF_LENGTH,USABLE_ROOF_WIDTH,PANELS_ALONG_LENGTH,PANELS_ALONG_WIDTH,POSSIBLE_PV_PANELS,WINDOW_AREA_SINGLE,ASPECT_RATIO,WINDOW_H,WINDOW_W,EXISTING_PV_PANELS
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055692,-4.223337,9/29/25,"RdSAP, existing dwelling",...,3.895189,2.754315,2.0,2.0,0,1.790829,1.75,1.011598,1.770297,0
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.052824,-4.220640,9/19/25,"RdSAP, existing dwelling",...,8.262944,3.895189,4.0,3.0,8,3.189657,2.25,1.190641,2.678942,0
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055503,-4.223691,7/29/25,"RdSAP, existing dwelling",...,4.956309,3.504640,2.0,3.0,2,2.044886,1.75,1.080975,1.891706,0
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.058427,-4.224838,7/22/25,"RdSAP, existing dwelling",...,8.295669,3.910616,4.0,3.0,8,3.295329,2.25,1.210203,2.722956,0
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.054738,-4.228307,7/17/25,"RdSAP, existing dwelling",...,10.616497,6.606565,5.0,5.0,21,2.763486,2.25,1.108249,2.493560,0


In [3]:
# Step 4: Validate Required Columns
# -------------------------------------------
required_columns = ["POSSIBLE_PV_PANELS", "EXISTING_PV_PANELS"]

missing_cols = [col for col in required_columns if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

In [4]:
# Step 5: Ensure Correct Data Types
# -------------------------------------------
df["POSSIBLE_PV_PANELS"] = pd.to_numeric(df["POSSIBLE_PV_PANELS"], errors="coerce").fillna(0).astype(int)
df["EXISTING_PV_PANELS"] = pd.to_numeric(df["EXISTING_PV_PANELS"], errors="coerce").fillna(0).astype(int)

In [5]:
# Step 6: Create MODELED_PV Column
# -------------------------------------------
df["MODELED_PV"] = df.apply(
    lambda row: row["EXISTING_PV_PANELS"] if row["EXISTING_PV_PANELS"] > 0 else row["POSSIBLE_PV_PANELS"],
    axis=1
)

In [6]:
# Step 7: Create PV_UPGRADE Column
# -------------------------------------------
df["PV_UPGRADE"] = df["EXISTING_PV_PANELS"].apply(lambda x: 0 if x > 0 else 1)

In [7]:
# Step 8: Save Updated Dataset
# -------------------------------------------
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_modeled_pv.csv"

df.to_csv(output_path, index=False)

print(f"File successfully saved to: {output_path}")

File successfully saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_modeled_pv.csv


In [8]:
# Step 9: Quick Sanity Check
# -------------------------------------------
df[["POSSIBLE_PV_PANELS", "EXISTING_PV_PANELS", "MODELED_PV", "PV_UPGRADE"]].head(10)

,POSSIBLE_PV_PANELS,EXISTING_PV_PANELS,MODELED_PV,PV_UPGRADE
0,0,0,0,1
1,8,0,8,1
2,2,0,2,1
3,8,0,8,1
4,21,0,21,1
5,5,0,5,1
6,5,0,5,1
7,11,0,11,1
8,11,9,9,0
9,14,21,21,0
